# Retrain on Colab

Rebuilds the model with the leak-free pipeline:
- imputation and frequency maps are fitted on train only
- the latest 20% of train is held out **before** SMOTE as validation
- Optuna tunes on validation, and the threshold is picked on validation
- the test set is scored once, at the end

Before running, add a Colab secret named `KAGGLE_API_TOKEN` (key icon in the left bar) and accept the competition rules at kaggle.com/c/ieee-fraud-detection/rules. A standard CPU runtime is enough; High-RAM helps.

In [1]:
!git clone -q https://github.com/Shashank17singh/Agentic-Fraud-Sentinel.git
%cd Agentic-Fraud-Sentinel
!git log --oneline -1
!pip install -q -r requirements.txt kaggle

/content/Agentic-Fraud-Sentinel
0770d09 (HEAD -> main, origin/main, origin/HEAD) Note that the README metrics predate the leak fixes


In [2]:
import os

from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!kaggle competitions download -c ieee-fraud-detection -p data/raw -q
!unzip -oq data/raw/ieee-fraud-detection.zip train_transaction.csv train_identity.csv -d data/raw
!ls -la data/raw

SecretNotFoundError: Secret KAGGLE_API_TOKEN does not exist.

In [ ]:
import sys

# Same layout as the local notebooks: outputs land in notebooks/data/, where the API looks.
%cd notebooks
sys.path.insert(0, "..")

from src.data.preprocessing import run_preprocessing_pipeline

X_train, X_val, X_test, Y_train, Y_val, Y_test = run_preprocessing_pipeline(
    raw_transaction_path="../data/raw/train_transaction.csv",
    raw_identity_path="../data/raw/train_identity.csv",
)

In [ ]:
from src.models.evaluate import evaluate_model, log_results, select_threshold
from src.models.train import train_xgboost_baseline

baseline = train_xgboost_baseline(X_train, Y_train)
baseline_results = evaluate_model(baseline, X_test, Y_test, model_name="xgb_baseline")
log_results(baseline_results)

In [ ]:
from src.models.train import tune_xgboost

tuned_xgb, study = tune_xgboost(X_train, Y_train, X_val, Y_val, n_trials=50)
print("Best validation PR-AUC:", study.best_value)
print("Best params:", study.best_params)

In [ ]:
# Threshold comes from validation only; test is scored once with it.
best_threshold = select_threshold(Y_val, tuned_xgb.predict_proba(X_val)[:, 1])
print(f"Threshold chosen on validation: {best_threshold:.4f}")

tuned_results = evaluate_model(tuned_xgb, X_test, Y_test, model_name="xgb_tuned")
log_results(tuned_results)
tuned_results_optimal = evaluate_model(
    tuned_xgb,
    X_test,
    Y_test,
    threshold=best_threshold,
    model_name="xgb_tuned_val_threshold",
)
log_results(tuned_results_optimal)

In [ ]:
import joblib

from src.explainability.shap_explainer import build_explainer, save_explainer

os.makedirs("data/models", exist_ok=True)
joblib.dump(
    {"model": tuned_xgb, "threshold": best_threshold, "metrics": tuned_results_optimal},
    "data/models/xgboost_production.pkl",
)
save_explainer(build_explainer(tuned_xgb, X_train.sample(1000, random_state=42)))

In [ ]:
# Download the new artifacts, then copy them over notebooks/data/ in your local repo.
!zip -qr /content/retrained.zip data/models data/model_results
from google.colab import files

files.download("/content/retrained.zip")